In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_DIR = Path("../ignore_files/data/full")
OUTPUT_DIR = Path("window_output_ctime_sliding50")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#TIME_COLUMN = "Arrival_Time"
#TIMESTAMP_UNITS_PER_SECOND = 1_000  # Arrival_Time is in milliseconds.

# If using Creation_Time, verify its units first.
# For nanosecond timestamps, use:
TIME_COLUMN = "Creation_Time"
TIMESTAMP_UNITS_PER_SECOND = 1_000_000_000

WINDOW_SECONDS = 2
WINDOW_TICKS = int(WINDOW_SECONDS * TIMESTAMP_UNITS_PER_SECOND)
STEP_TICKS = WINDOW_TICKS // 2  # 50% overlap: 2-second windows, 1-second step

GROUP_COLUMNS = ["User", "Model", "Device"]
AXES = ["x", "y", "z"]
SIGNAL_COLUMNS = AXES + ["magnitude"]

# Provisional quality thresholds, NOT dataset-validated defaults.
# Review the audit report before treating the output as training data.
MIN_SAMPLES_PER_SENSOR = 2
MIN_TIME_COVERAGE_RATIO = 0.80

# Optional internal-gap filter
MAX_INTERNAL_GAP_SECONDS = 0.1

# This implementation assumes that separate recording sessions do
# not overlap on the selected timestamp axis within a user/device.
# Add a shared session identifier to GROUP_COLUMNS if necessary.

FILES = {
    "acc": [
        "Watch_accelerometer.csv",
        "Phones_accelerometer.csv",
    ],
    "gyro": [
        "Watch_gyroscope.csv",
        "Phones_gyroscope.csv",
    ],
}

REQUIRED_COLUMNS = list(
    dict.fromkeys(GROUP_COLUMNS + [TIME_COLUMN, "gt"] + AXES)
)

started = time.perf_counter()

print("=" * 72)
print("WINDOW-BASED SENSOR FEATURE EXTRACTION")
print("=" * 72)
print(f"Timestamp column: {TIME_COLUMN}")
print(f"Assumed timestamp units per second: {TIMESTAMP_UNITS_PER_SECOND:,}")
print(f"Window duration: {WINDOW_SECONDS} seconds")
print("Overlap: 50%")
print(f"Grouping columns: {GROUP_COLUMNS}")
print(f"Minimum samples per sensor: {MIN_SAMPLES_PER_SENSOR}")
print(f"Minimum timestamp span / window duration: {MIN_TIME_COVERAGE_RATIO:.0%}")
print(f"Maximum internal gap: {MAX_INTERNAL_GAP_SECONDS}")
print()
print("IMPORTANT:")
print("- Timestamp units are configured, not automatically verified.")
print("- Matching windows do not prove physical sensor synchronization.")
print("- Quality thresholds are provisional.")
print("- No interpolation or raw-sample matching is performed.")
print("- Windows use shared fixed boundaries, not separate sensor start times.")


# ============================================================
# 2. LOAD ORIGINAL SENSOR FILES
# ============================================================

def load_sensor_files(filenames, sensor):
    frames = []

    for filename in filenames:
        path = DATA_DIR / filename
        print(f"\nLoading {sensor}: {path}")

        frame = pd.read_csv(path, usecols=REQUIRED_COLUMNS)
        frames.append(frame)

        print(f"  Rows loaded: {len(frame):,}")

    data = pd.concat(frames, ignore_index=True)

    print(f"Total {sensor} rows: {len(data):,}")
    return data


df_acc = load_sensor_files(FILES["acc"], "accelerometer")
df_gyro = load_sensor_files(FILES["gyro"], "gyroscope")


# ============================================================
# 3. VALIDATE INPUT
# ============================================================

def validate_input(data, sensor):
    missing_keys = data[GROUP_COLUMNS + [TIME_COLUMN]].isna().any(axis=1)

    if missing_keys.any():
        raise ValueError(
            f"{sensor}: {missing_keys.sum():,} rows have missing "
            "group identifiers or timestamps."
        )

    if not pd.api.types.is_integer_dtype(data[TIME_COLUMN]):
        raise TypeError(
            f"{sensor}: {TIME_COLUMN} must contain integer timestamps. "
            "Inspect the column before converting it."
        )

    for axis in AXES:
        if not pd.api.types.is_numeric_dtype(data[axis]):
            raise TypeError(f"{sensor}: axis '{axis}' is not numeric.")

    invalid_axes = ~np.isfinite(data[AXES].to_numpy()).all(axis=1)

    print(f"\n{sensor.upper()} INPUT CHECK")
    print(f"  Rows: {len(data):,}")
    print(f"  Users: {data['User'].nunique():,}")
    print(f"  Groups: {len(data[GROUP_COLUMNS].drop_duplicates()):,}")
    print(f"  Rows with non-finite axis values: {invalid_axes.sum():,}")
    print(f"  Minimum timestamp: {data[TIME_COLUMN].min()}")
    print(f"  Maximum timestamp: {data[TIME_COLUMN].max()}")

    # Non-finite signal values are retained for window assignment.
    # Any window containing them will be rejected later.


validate_input(df_acc, "acc")
validate_input(df_gyro, "gyro")

input_groups = (
    pd.concat(
        [df_acc[GROUP_COLUMNS], df_gyro[GROUP_COLUMNS]],
        ignore_index=True,
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"\nTotal unique input groups: {len(input_groups):,}")


# ============================================================
# 4. BUILD FEATURES AND QUALITY METADATA PER SENSOR
# ============================================================

def build_windows(data, sensor):
    print(f"\nBuilding {sensor} windows...")
    step_started = time.perf_counter()

    # Each window k covers:
    # [k * STEP_TICKS, k * STEP_TICKS + WINDOW_TICKS)

    data = data.copy()
    base_window_id = data[TIME_COLUMN] // STEP_TICKS

    current = data.assign(window_id=base_window_id)
    previous = data.assign(window_id=base_window_id - 1)

    data = pd.concat([current, previous], ignore_index=True)

    del current, previous

    labels = data["gt"].astype("string").str.strip().str.lower()
    data["label"] = labels.mask(
        labels.isin(["", "null", "nan", "none", "na"])
    )

    data["invalid_axes"] = ~np.isfinite(
        data[AXES].to_numpy()
    ).all(axis=1)

    data["magnitude"] = np.sqrt(
        data["x"] ** 2 + data["y"] ** 2 + data["z"] ** 2
    )

    keys = GROUP_COLUMNS + ["window_id"]
    data = data.sort_values(keys + [TIME_COLUMN])

    grouped = data.groupby(keys, observed=True, sort=False)

    # Internal timestamp gaps only: no differences across windows.
    data["gap_seconds"] = (
        grouped[TIME_COLUMN].diff()
        / TIMESTAMP_UNITS_PER_SECOND
    )

    grouped = data.groupby(keys, observed=True, sort=False)

    features = grouped[SIGNAL_COLUMNS].agg(
        ["mean", "std", "min", "max"]
    )
    features.columns = [
        f"{sensor}_{signal}_{stat}"
        for signal, stat in features.columns
    ]

    quality = grouped.agg(
        n=(TIME_COLUMN, "size"),
        unique_times=(TIME_COLUMN, "nunique"),
        first_time=(TIME_COLUMN, "min"),
        last_time=(TIME_COLUMN, "max"),
        max_gap_seconds=("gap_seconds", "max"),
        invalid_axis_rows=("invalid_axes", "sum"),
        label_count=("label", "nunique"),
        labeled_n=("label", "count"),
        label=("label", "first"),
    )

    quality["span_seconds"] = (
        quality["last_time"] - quality["first_time"]
    ) / TIMESTAMP_UNITS_PER_SECOND

    # This is a timestamp-span proxy, not proof of continuous coverage.
    quality["span_ratio"] = (
        quality["span_seconds"] / WINDOW_SECONDS
    )

    quality["duplicate_timestamp_fraction"] = (
        1 - quality["unique_times"] / quality["n"]
    )

    quality["valid_label"] = (
        quality["label_count"].eq(1)
        & quality["labeled_n"].eq(quality["n"])
    )

    result = features.join(
        quality.add_prefix(f"{sensor}_")
    ).reset_index()

    print(f"  Windows created: {len(result):,}")
    print(f"  Time taken: {time.perf_counter() - step_started:.1f} seconds")

    return result, list(features.columns)


acc_windows, acc_feature_columns = build_windows(df_acc, "acc")
gyro_windows, gyro_feature_columns = build_windows(df_gyro, "gyro")

feature_columns = acc_feature_columns + gyro_feature_columns
window_keys = GROUP_COLUMNS + ["window_id"]


# ============================================================
# 5. JOIN WINDOW FEATURES, NOT RAW SENSOR ROWS
# ============================================================

print("\nJoining sensor windows...")

audit = acc_windows.merge(
    gyro_windows,
    on=window_keys,
    how="outer",
    validate="one_to_one",
    indicator="sensor_presence",
)

print(f"Total observed windows: {len(audit):,}")
print(audit["sensor_presence"].value_counts().to_string())


# ============================================================
# 6. APPLY EXPLICIT QUALITY RULES
# ============================================================

audit["both_sensors"] = audit["sensor_presence"].eq("both")

audit["labels_valid"] = (
    audit["acc_valid_label"].eq(True)
    & audit["gyro_valid_label"].eq(True)
)

audit["labels_agree"] = (
    audit["acc_label"]
    .eq(audit["gyro_label"])
    .fillna(False)
)

audit["axis_values_valid"] = (
    audit["acc_invalid_axis_rows"].eq(0)
    & audit["gyro_invalid_axis_rows"].eq(0)
)

audit["enough_samples"] = (
    audit["acc_n"].ge(MIN_SAMPLES_PER_SENSOR)
    & audit["gyro_n"].ge(MIN_SAMPLES_PER_SENSOR)
)

audit["enough_time_span"] = (
    audit["acc_span_ratio"].ge(MIN_TIME_COVERAGE_RATIO)
    & audit["gyro_span_ratio"].ge(MIN_TIME_COVERAGE_RATIO)
)

if MAX_INTERNAL_GAP_SECONDS is None:
    audit["internal_gaps_ok"] = True
else:
    audit["internal_gaps_ok"] = (
        audit["acc_max_gap_seconds"].le(MAX_INTERNAL_GAP_SECONDS)
        & audit["gyro_max_gap_seconds"].le(MAX_INTERNAL_GAP_SECONDS)
    )

audit["finite_features"] = np.isfinite(
    audit[feature_columns].to_numpy(dtype=float)
).all(axis=1)

# Assign one rejection reason per window, in this priority order.
# A window can violate multiple rules; the individual flags remain
# available in the audit file.
audit["status"] = np.select(
    [
        ~audit["both_sensors"],
        ~audit["labels_valid"],
        ~audit["labels_agree"],
        ~audit["axis_values_valid"],
        ~audit["enough_samples"],
        ~audit["enough_time_span"],
        ~audit["internal_gaps_ok"],
        ~audit["finite_features"],
    ],
    [
        "missing_sensor",
        "missing_or_mixed_activity",
        "sensor_activity_disagreement",
        "invalid_axis_values",
        "too_few_samples",
        "insufficient_timestamp_span",
        "large_internal_gap",
        "non_finite_features",
    ],
    default="accepted",
)

accepted = audit.loc[audit["status"].eq("accepted")].copy()
accepted["gt"] = accepted["acc_label"]

# Keep identifiers for splitting and inspection, not as model features.
dataset = accepted[
    window_keys + feature_columns + ["gt"]
].copy()


# ============================================================
# 7. PRINT DIAGNOSTIC REPORTS
# ============================================================

print("\n" + "=" * 72)
print("WINDOW PROCESSING SUMMARY")
print("=" * 72)

status_counts = audit["status"].value_counts().rename_axis("status")
print(status_counts.to_string())

acceptance_rate = (
    len(dataset) / len(audit) if len(audit) else 0.0
)

print(f"\nAccepted windows: {len(dataset):,} / {len(audit):,}")
print(f"Acceptance rate: {acceptance_rate:.2%}")
print(f"Features per window: {len(feature_columns)}")

quality_columns = [
    f"{sensor}_{field}"
    for sensor in ["acc", "gyro"]
    for field in [
        "n",
        "unique_times",
        "span_seconds",
        "span_ratio",
        "max_gap_seconds",
        "duplicate_timestamp_fraction",
    ]
]

print("\nQUALITY BEFORE FILTERING: WINDOWS WITH BOTH SENSORS")
both_sensor_windows = audit.loc[audit["both_sensors"]]

if both_sensor_windows.empty:
    print("No windows contain both sensors.")
else:
    print(
        both_sensor_windows[quality_columns]
        .describe(percentiles=[0.05, 0.50, 0.95])
        .round(3)
        .to_string()
    )

# Model-level retention includes rejected windows.
model_report = pd.crosstab(audit["Model"], audit["status"])
model_report["total_windows"] = model_report.sum(axis=1)

if "accepted" not in model_report:
    model_report["accepted"] = 0

model_report["acceptance_pct"] = (
    100 * model_report["accepted"] / model_report["total_windows"]
)

print("\nWINDOW RETENTION BY MODEL")
print(model_report.round(2).to_string())

# Include input groups with zero accepted windows.
observed_counts = (
    audit.groupby(GROUP_COLUMNS, observed=True)
    .size()
    .rename("observed_windows")
)

accepted_counts = (
    dataset.groupby(GROUP_COLUMNS, observed=True)
    .size()
    .rename("accepted_windows")
)

group_report = (
    input_groups.set_index(GROUP_COLUMNS)
    .join(observed_counts)
    .join(accepted_counts)
    .fillna(0)
    .astype("int64")
    .reset_index()
)

print("\nGROUP COVERAGE")
print(f"Input groups: {len(group_report):,}")
print(
    "Groups with accepted windows: "
    f"{group_report['accepted_windows'].gt(0).sum():,}"
)
print(
    "Groups with no accepted windows: "
    f"{group_report['accepted_windows'].eq(0).sum():,}"
)

activity_report = pd.crosstab(
    dataset["Model"],
    dataset["gt"],
)

print("\nACCEPTED WINDOWS BY MODEL AND ACTIVITY")
if dataset.empty:
    print("No windows passed all filters.")
    print("Inspect the rejection reasons before changing thresholds.")
else:
    print(activity_report.to_string())

    print("\nACCEPTED WINDOWS BY ACTIVITY")
    print(dataset["gt"].value_counts().to_string())

    print("\nQUALITY AFTER FILTERING")
    print(
        accepted[quality_columns]
        .describe(percentiles=[0.05, 0.50, 0.95])
        .round(3)
        .to_string()
    )


# ============================================================
# 8. SAVE DATASET, AUDIT, REPORTS AND CONFIGURATION
# ============================================================

# Omit feature values from the audit to avoid saving them twice.
audit_columns = [
    column for column in audit.columns
    if column not in feature_columns
]

config = {
    "data_directory": str(DATA_DIR),
    "input_files": FILES,
    "timestamp_column": TIME_COLUMN,
    "timestamp_units_per_second": TIMESTAMP_UNITS_PER_SECOND,
    "window_seconds": WINDOW_SECONDS,
    "overlap_fraction": 0.5,
    "step_seconds": STEP_TICKS / TIMESTAMP_UNITS_PER_SECOND,
    "group_columns": GROUP_COLUMNS,
    "minimum_samples_per_sensor": MIN_SAMPLES_PER_SENSOR,
    "minimum_time_span_ratio": MIN_TIME_COVERAGE_RATIO,
    "maximum_internal_gap_seconds": MAX_INTERNAL_GAP_SECONDS,
    "feature_columns": feature_columns,
    "target_column": "gt",
    "split_group_column": "User",
    "input_acc_rows": len(df_acc),
    "input_gyro_rows": len(df_gyro),
    "input_groups": len(input_groups),
    "observed_windows": len(audit),
    "accepted_windows": len(dataset),
    "timestamp_units_automatically_verified": False,
    "physical_sensor_synchronization_verified": False,
}

dataset.to_csv(OUTPUT_DIR / "window_features.csv", index=False)
audit[audit_columns].to_csv(
    OUTPUT_DIR / "window_quality_audit.csv",
    index=False,
)
status_counts.rename("windows").to_csv(
    OUTPUT_DIR / "processing_summary.csv"
)
model_report.to_csv(OUTPUT_DIR / "retention_by_model.csv")
group_report.to_csv(OUTPUT_DIR / "retention_by_group.csv", index=False)
activity_report.to_csv(OUTPUT_DIR / "activities_by_model.csv")

with open(OUTPUT_DIR / "configuration.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)

print("\n" + "=" * 72)
print("PROCESSING COMPLETE")
print("=" * 72)
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Elapsed time: {time.perf_counter() - started:.1f} seconds")
print()
print("NEXT STEPS:")
print("1. Inspect rejection rates and quality metrics by model.")
print("2. Verify timestamp units, recording sessions and sensor alignment.")
print("3. Use only the configured feature_columns as model inputs.")
print("4. Use gt as the target and User for participant-based splitting.")
print("5. Do not use User, Device, Model or window_id as baseline features.")

WINDOW-BASED SENSOR FEATURE EXTRACTION
Timestamp column: Creation_Time
Assumed timestamp units per second: 1,000,000,000
Window duration: 2 seconds
Overlap: 50%
Grouping columns: ['User', 'Model', 'Device']
Minimum samples per sensor: 2
Minimum timestamp span / window duration: 80%
Maximum internal gap: 0.1

IMPORTANT:
- Timestamp units are configured, not automatically verified.
- Matching windows do not prove physical sensor synchronization.
- Quality thresholds are provisional.
- No interpolation or raw-sample matching is performed.
- Windows use shared fixed boundaries, not separate sensor start times.

Loading accelerometer: ..\ignore_files\data\full\Watch_accelerometer.csv
  Rows loaded: 3,540,962

Loading accelerometer: ..\ignore_files\data\full\Phones_accelerometer.csv
  Rows loaded: 13,062,475
Total accelerometer rows: 16,603,437

Loading gyroscope: ..\ignore_files\data\full\Watch_gyroscope.csv
  Rows loaded: 3,205,431

Loading gyroscope: ..\ignore_files\data\full\Phones_gyros

In [2]:
print("Accelerometer rows by model:")
print(df_acc.groupby("Model").size())

print("\nGyroscope rows by model:")
print(df_gyro.groupby("Model").size())

Accelerometer rows by model:
Model
gear           449077
lgwatch       3091885
nexus4        6240983
s3            2959749
s3mini        1635179
samsungold    2226564
dtype: int64

Gyroscope rows by model:
Model
gear        447506
lgwatch    2757925
nexus4     6241005
s3         6056439
s3mini     1635188
dtype: int64


In [3]:
inspection_columns = window_keys + [
    "status",
    "acc_n",
    "gyro_n",
    "acc_span_seconds",
    "gyro_span_seconds",
    "acc_max_gap_seconds",
    "gyro_max_gap_seconds",
    "acc_duplicate_timestamp_fraction",
    "gyro_duplicate_timestamp_fraction",
]

print("\nHighest accelerometer sample counts:")
display(
    audit.nlargest(10, "acc_n")[inspection_columns]
)

print(f"\nLargest accelerometer gaps on {TIME_COLUMN}:")
display(
    audit.nlargest(10, "acc_max_gap_seconds")[inspection_columns]
)

print(f"\nLargest gyroscope gaps on {TIME_COLUMN}:")
display(
    audit.nlargest(10, "gyro_max_gap_seconds")[inspection_columns]
)


Highest accelerometer sample counts:


,User,Model,Device,window_id,status,acc_n,gyro_n,acc_span_seconds,gyro_span_seconds,acc_max_gap_seconds,gyro_max_gap_seconds,acc_duplicate_timestamp_fraction,gyro_duplicate_timestamp_fraction
38316,c,nexus4,nexus4_2,1424696225,accepted,579.0,579.0,1.960036,1.959737,0.008026,0.007416,0.0,0.0
88893,f,nexus4,nexus4_2,1424782789,accepted,579.0,579.0,1.963865,1.965635,0.005158,0.006439,0.0,0.0
118239,h,nexus4,nexus4_2,1424702036,missing_or_mixed_activity,578.0,578.0,1.919856,1.919856,0.007874,0.007294,0.0,0.0
70116,e,nexus4,nexus4_2,1424790431,accepted,577.0,577.0,1.993281,1.993281,0.007904,0.007202,0.0,0.0
103054,g,nexus4,nexus4_2,1424690649,accepted,574.0,574.0,1.993757,1.993757,0.008026,0.007263,0.0,0.0
37497,c,nexus4,nexus4_1,1424695396,accepted,572.0,572.0,1.968572,1.968603,0.008270,0.007690,0.0,0.0
129934,i,nexus4,nexus4_1,1424777414,missing_or_mixed_activity,571.0,571.0,1.995558,1.995589,0.007751,0.007202,0.0,0.0
116510,h,nexus4,nexus4_1,1424700160,accepted,565.0,564.0,1.996536,1.991500,0.007843,0.007263,0.0,0.0
118663,h,nexus4,nexus4_2,1424702526,accepted,561.0,561.0,1.994950,1.995011,0.005147,0.005188,0.0,0.0
99878,g,nexus4,nexus4_1,1424687558,missing_or_mixed_activity,559.0,559.0,1.993217,1.993217,0.005157,0.005218,0.0,0.0



Largest accelerometer gaps on Creation_Time:


,User,Model,Device,window_id,status,acc_n,gyro_n,acc_span_seconds,gyro_span_seconds,acc_max_gap_seconds,gyro_max_gap_seconds,acc_duplicate_timestamp_fraction,gyro_duplicate_timestamp_fraction
63636,e,gear,gear_2,1655,missing_or_mixed_activity,4.0,2.0,1.963255,0.009847,1.943211,0.009847,0.000000,0.0
97988,g,gear,gear_1,25848,large_internal_gap,12.0,10.0,1.991154,1.991162,1.895431,1.907070,0.000000,0.0
350,a,gear,gear_1,28274,missing_or_mixed_activity,13.0,12.0,1.980419,1.980425,1.869766,1.889881,0.000000,0.0
167,a,gear,gear_1,28087,missing_or_mixed_activity,15.0,14.0,1.992629,1.992635,1.863162,1.873354,0.000000,0.0
292,a,gear,gear_1,28215,large_internal_gap,22.0,21.0,1.985330,1.985324,1.800297,1.813934,0.000000,0.0
67802,e,nexus4,nexus4_1,1424788319,large_internal_gap,45.0,45.0,1.994048,1.994079,1.777496,1.777496,0.000000,0.0
63613,e,gear,gear_2,1631,large_internal_gap,15.0,14.0,1.893480,1.893480,1.774339,1.774356,0.066667,0.0
33393,c,gear,gear_2,1153,missing_or_mixed_activity,25.0,24.0,1.990808,1.990814,1.762030,1.771842,0.000000,0.0
6425,a,nexus4,nexus4_2,1424700205,large_internal_gap,44.0,44.0,1.968800,1.973805,1.757324,1.762349,0.000000,0.0
118167,h,nexus4,nexus4_2,1424701941,missing_or_mixed_activity,55.0,56.0,1.986237,1.993378,1.722259,1.722259,0.000000,0.0



Largest gyroscope gaps on Creation_Time:


,User,Model,Device,window_id,status,acc_n,gyro_n,acc_span_seconds,gyro_span_seconds,acc_max_gap_seconds,gyro_max_gap_seconds,acc_duplicate_timestamp_fraction,gyro_duplicate_timestamp_fraction
97988,g,gear,gear_1,25848,large_internal_gap,12.0,10.0,1.991154,1.991162,1.895431,1.907070,0.000000,0.0
350,a,gear,gear_1,28274,missing_or_mixed_activity,13.0,12.0,1.980419,1.980425,1.869766,1.889881,0.000000,0.0
167,a,gear,gear_1,28087,missing_or_mixed_activity,15.0,14.0,1.992629,1.992635,1.863162,1.873354,0.000000,0.0
292,a,gear,gear_1,28215,large_internal_gap,22.0,21.0,1.985330,1.985324,1.800297,1.813934,0.000000,0.0
67802,e,nexus4,nexus4_1,1424788319,large_internal_gap,45.0,45.0,1.994048,1.994079,1.777496,1.777496,0.000000,0.0
63613,e,gear,gear_2,1631,large_internal_gap,15.0,14.0,1.893480,1.893480,1.774339,1.774356,0.066667,0.0
33393,c,gear,gear_2,1153,missing_or_mixed_activity,25.0,24.0,1.990808,1.990814,1.762030,1.771842,0.000000,0.0
6425,a,nexus4,nexus4_2,1424700205,large_internal_gap,44.0,44.0,1.968800,1.973805,1.757324,1.762349,0.000000,0.0
39114,c,nexus4,nexus4_2,1424697105,large_internal_gap,56.0,56.0,1.994451,1.994512,1.722198,1.722539,0.000000,0.0
118167,h,nexus4,nexus4_2,1424701941,missing_or_mixed_activity,55.0,56.0,1.986237,1.993378,1.722259,1.722259,0.000000,0.0


In [4]:
from pathlib import Path

import pandas as pd

data_dir = Path("../ignore_files/data/full")

target_user = "g"
target_model = "nexus4"
target_device = "nexus4_2"
target_window_id = 712344195

window_ms = 2_000
context_ms = 2_000

start_ms = target_window_id * window_ms
end_ms = start_ms + window_ms

columns = [
    "User", "Model", "Device",
    "Arrival_Time", "Creation_Time",
    "x", "y", "z", "gt",
]


def inspect_original_rows(filename, sensor):
    parts = []

    # Read in chunks to avoid loading the entire original file again.
    for chunk in pd.read_csv(
        data_dir / filename,
        usecols=columns,
        chunksize=500_000,
    ):
        mask = (
            chunk["User"].eq(target_user)
            & chunk["Model"].eq(target_model)
            & chunk["Device"].eq(target_device)
            & chunk["Arrival_Time"].ge(start_ms - context_ms)
            & chunk["Arrival_Time"].lt(end_ms + context_ms)
        )

        if mask.any():
            parts.append(chunk.loc[mask].copy())

    if not parts:
        print(f"\n{sensor}: no matching rows.")
        return pd.DataFrame()

    rows = pd.concat(parts, ignore_index=True)

    # Arrival sorting makes the observed delivery gap visible.
    # Creation_Time is a tie-breaker for identical arrival timestamps.
    rows = rows.sort_values(
        ["Arrival_Time", "Creation_Time"]
    ).reset_index(drop=True)

    rows["inside_target_window"] = (
        rows["Arrival_Time"].ge(start_ms)
        & rows["Arrival_Time"].lt(end_ms)
    )

    rows["arrival_relative_ms"] = rows["Arrival_Time"] - start_ms
    rows["arrival_gap_ms"] = rows["Arrival_Time"].diff()

    # Keep creation differences in raw units until units are verified.
    # These are differences in arrival-sorted order.
    creation_delta = (
        rows["Creation_Time"]
        - rows["Creation_Time"].shift(1, fill_value=0)
    ).astype("Int64")

    creation_delta.iloc[0] = pd.NA
    rows["creation_delta_raw"] = creation_delta

    # Only after confirming that the units are nanoseconds:
    rows["creation_gap_ms"] = rows["creation_delta_raw"] / 1_000_000

    print(f"\n=== {sensor.upper()} ===")
    print(f"Target arrival interval: [{start_ms}, {end_ms})")
    print(
        "Rows inside target window:",
        int(rows["inside_target_window"].sum()),
    )

    print("\nRows inside the target window:")
    print(
        rows.loc[rows["inside_target_window"]].to_string(index=False)
    )

    print("\nLargest arrival gaps, including surrounding context:")
    print(
        rows.nlargest(5, "arrival_gap_ms")[
            [
                "Arrival_Time",
                "Creation_Time",
                "inside_target_window",
                "arrival_relative_ms",
                "arrival_gap_ms",
                "creation_delta_raw",
                "gt",
            ]
        ].to_string(index=False)
    )

    return rows


acc_inspection = inspect_original_rows(
    "Phones_accelerometer.csv", "accelerometer"
)

gyro_inspection = inspect_original_rows(
    "Phones_gyroscope.csv", "gyroscope"
)


=== ACCELEROMETER ===
Target arrival interval: [1424688390000, 1424688392000)
Rows inside target window: 10

Rows inside the target window:
 Arrival_Time       Creation_Time         x         y         z User  Model   Device       gt  inside_target_window  arrival_relative_ms  arrival_gap_ms  creation_delta_raw  creation_gap_ms
1424688390001 1424690236056446917 -5.364121  1.074127  9.883530    g nexus4 nexus4_2 stairsup                  True                    1             5.0             5035401         5.035401
1424688390006 1424690236061482317 -5.283188  1.445465 10.173935    g nexus4 nexus4_2 stairsup                  True                    6             5.0             5035400           5.0354
1424688390011 1424690236066517717 -5.178451  1.700165 10.384598    g nexus4 nexus4_2 stairsup                  True                   11             5.0             5035400           5.0354
1424688390016 1424690236071553118 -5.126083  1.964386 10.790451    g nexus4 nexus4_2 stairsup      

In [5]:
lgwatch_missing = (
    audit.loc[
        audit["Model"].eq("lgwatch")
        & audit["sensor_presence"].ne("both")
    ]
    .groupby(
        ["User", "Model", "Device", "sensor_presence"],
        observed=True,
    )
    .size()
    .rename("windows")
    .reset_index()
    .sort_values("windows", ascending=False)
)

display(lgwatch_missing)

,User,Model,Device,sensor_presence,windows
12,g,lgwatch,lgwatch_2,left_only,491
7,e,lgwatch,lgwatch_2,left_only,385
9,f,lgwatch,lgwatch_2,left_only,337
0,a,lgwatch,lgwatch_2,left_only,334
3,c,lgwatch,lgwatch_2,left_only,277
19,i,lgwatch,lgwatch_2,left_only,263
17,i,lgwatch,lgwatch_1,left_only,186
14,h,lgwatch,lgwatch_1,left_only,160
8,e,lgwatch,lgwatch_2,right_only,156
13,g,lgwatch,lgwatch_2,right_only,152


In [6]:
print("Accepted windows by participant and activity:")
display(pd.crosstab(dataset["User"], dataset["gt"]))

print("\nAccepted windows by participant and model:")
display(pd.crosstab(dataset["User"], dataset["Model"]))

X = dataset[feature_columns]
y = dataset["gt"]
groups = dataset["User"]

Accepted windows by participant and activity:


gt,bike,sit,stairsdown,stairsup,stand,walk
User,,,,,,
a,1238,1846,1168,1295,1703,1521
b,1861,1437,1206,1493,1369,1962
c,1059,1637,1132,1475,1323,1727
d,1180,1664,997,1473,1604,1914
e,1725,1715,1157,1282,1668,1957
f,1349,1311,1132,1545,1289,1516
g,956,1429,1027,998,1298,1633
h,1054,1561,1217,1289,965,1483
i,1252,1462,975,1039,1213,1594



Accepted windows by participant and model:


Model,gear,lgwatch,nexus4,s3,s3mini
User,,,,,
a,222,1694,2686,2798,1371
b,210,1873,2897,2898,1450
c,241,1747,2518,2617,1230
d,200,1670,2835,2763,1364
e,271,1905,2919,2956,1453
f,361,1860,2928,1417,1576
g,182,75,2721,2881,1482
h,845,11,2589,2714,1410
i,0,32,2993,2988,1522


In [7]:
# One row per user/device, including every processing status.
watch_retention = pd.crosstab(
    index=[
        audit["User"],
        audit["Model"],
        audit["Device"],
    ],
    columns=audit["status"],
)

watch_retention = watch_retention.loc[
    watch_retention.index.get_level_values("Model").isin(
        ["gear", "lgwatch"]
    )
].copy()

watch_retention["total_observed_windows"] = watch_retention.sum(axis=1)

if "accepted" not in watch_retention.columns:
    watch_retention["accepted"] = 0

watch_retention["accepted_pct"] = (
    100
    * watch_retention["accepted"]
    / watch_retention["total_observed_windows"]
)

display(watch_retention.round(2))

status                  accepted  insufficient_timestamp_span  \
User Model   Device                                             
a    gear    gear_1          178                          123   
             gear_2           44                           61   
     lgwatch lgwatch_1      1642                            4   
             lgwatch_2        52                           25   
b    gear    gear_1          197                           16   
             gear_2           13                           67   
     lgwatch lgwatch_1      1873                            4   
             lgwatch_2         0                            0   
c    gear    gear_1          209                          103   
             gear_2           32                           70   
     lgwatch lgwatch_1      1687                            3   
             lgwatch_2        60                            7   
d    gear    gear_1          182                           78   
             gear_2           18                           60   
     lgwatch lgwatch_1      1624                            4   
             lgwatch_2        46                           12   
e    gear    gear_1          240                           15   
             gear_2           31                           59   
     lgwatch lgwatch_1      1782                            2   
             lgwatch_2       123                           16   
f    gear    gear_1          308                           21   
             gear_2           53                           73   
     lgwatch lgwatch_1      1838                            3   
             lgwatch_2        22                           11   
g    gear    gear_1          176                          119   
             gear_2            6                           67   
     lgwatch lgwatch_1         0                            0   
             lgwatch_2        75                           34   
h    gear    gear_1          843                           18   
             gear_2            2                           67   
     lgwatch lgwatch_1        11                           10   
             lgwatch_2         0                            0   
i    gear    gear_2            0                            0   
     lgwatch lgwatch_1        13                            3   
             lgwatch_2        19                            9   

status                  large_internal_gap  missing_or_mixed_activity  \
User Model   Device                                                     
a    gear    gear_1                     68                         77   
             gear_2                     15                         53   
     lgwatch lgwatch_1                   0                        240   
             lgwatch_2                   3                          0   
b    gear    gear_1                    153                        101   
             gear_2                      1                         40   
     lgwatch lgwatch_1                   0                        363   
             lgwatch_2                   0                          0   
c    gear    gear_1                     89                         94   
             gear_2                     10                        108   
     lgwatch lgwatch_1                   0                        244   
             lgwatch_2                   0                          9   
d    gear    gear_1                    112                         65   
             gear_2                      7                         29   
     lgwatch lgwatch_1                   0                        201   
             lgwatch_2                   0                          0   
e    gear    gear_1                    145                        154   
             gear_2                     16                         53   
     lgwatch lgwatch_1                   0                        489   
             lgwatch_2                   0                   

In [8]:
keys = ["User", "Model", "Device"]


def creation_ranges(df, sensor):
    subset = df.loc[df["Model"].eq("lgwatch")]

    ranges = (
        subset.groupby(["User", "Model", "Device"])["Creation_Time"]
        .agg(["min", "max", "size"])
        .astype("Int64")
        .rename(columns={
            "min": f"{sensor}_start",
            "max": f"{sensor}_end",
            "size": f"{sensor}_rows",
        })
    )

    return ranges


range_report = creation_ranges(df_acc, "acc").join(
    creation_ranges(df_gyro, "gyro"),
    how="outer",
)

# Differences are calculated before conversion to seconds.
range_report["start_difference_seconds"] = (
    range_report["gyro_start"] - range_report["acc_start"]
) / 1_000_000_000

range_report["end_difference_seconds"] = (
    range_report["gyro_end"] - range_report["acc_end"]
) / 1_000_000_000

display(range_report)

acc_start          acc_end  acc_rows  \
User Model   Device                                                  
a    lgwatch lgwatch_1  207817520343346  209701035294082    381243   
             lgwatch_2  123555435065228  125435396202931     72603   
b    lgwatch lgwatch_1  295831887411248  298069335316564    453215   
             lgwatch_2  213514462023901  213514462023901         1   
c    lgwatch lgwatch_1  204760978091936  206692756624057    391505   
             lgwatch_2  120631960305262  122565067305740     64205   
d    lgwatch lgwatch_1  292859876523215  294686601275166    369876   
             lgwatch_2  209313263050624  210427130928916     32321   
e    lgwatch lgwatch_1  298372334757501  300643638599590    459446   
             lgwatch_2  214089493023240  216525986027449    104883   
f    lgwatch lgwatch_1  290151306028670  292355384188379    445594   
             lgwatch_2  205897932297919  208187484598470     72350   
g    lgwatch lgwatch_1  200228355692555  200228355692555         1   
             lgwatch_2  113724895094724  116107389038747    113181   
h    lgwatch lgwatch_1  210715022425903  212089277303485     34919   
             lgwatch_2  127880178351576  127880178351576         1   
i    lgwatch lgwatch_1  287096247623490  289237941459302     41015   
             lgwatch_2  203221023933775  205449771092299     55526   

                             gyro_start         gyro_end  gyro_rows  \
User Model   Device                                                   
a    lgwatch lgwatch_1  207817520343346  209701035294082     381243   
             lgwatch_2  123622660986859  125475649987110      29961   
b    lgwatch lgwatch_1  295831887411248  298069335316564     453215   
             lgwatch_2             <NA>             <NA>       <NA>   
c    lgwatch lgwatch_1  204760978091936  206692756624057     391505   
             lgwatch_2  120674276284266  122565067305740      35116   
d    lgwatch lgwatch_1  292859871579368  294686601275166     369877   
             lgwatch_2  209355539021083  210432397330876      12625   
e    lgwatch lgwatch_1  298372334757501  300643638599590     459446   
             lgwatch_2  214156892071091  216525986027449      52196   
f    lgwatch lgwatch_1  290151306028670  292355384188379     445594   
             lgwatch_2  205959575597878  208223865370500      28676   
g    lgwatch lgwatch_1             <NA>             <NA>       <NA>   
             lgwatch_2  113775829906244  116122655396169      44717   
h    lgwatch lgwatch_1  210757058436645  212089277303485      10052   
             lgwatch_2             <NA>             <NA>       <NA>   
i    lgwatch lgwatch_1  287138180881547  289296391959790      16280   
             lgwatch_2  203300875923521  205519508530829      27422   

                        start_difference_seconds  end_difference_seconds  
User Model   Device                                                       
a    lgwatch lgwatch_1                       0.0                     0.0  
             lgwatch_2                 67.225922               40.253784  
b    lgwatch lgwatch_1                       0.0                     0.0  
             lgwatch_2                      <NA>                    <NA>  
c    lgwatch lgwatch_1                       0.0                     0.0  
             lgwatch_2                 42.315979                     0.0  
d    lgwatch lgwatch_1                 -0.004944                     0.0  
             lgwatch_2                  42.27597                5.266402  
e    lgwatch lgwatch_1                       0.0                     0.0  
             lgwatch_2                 67.399048                     0.0  
f    lgwatch lgwatch_1                       0.0                     0.0  
             lgwatch_2                   61.6433               36.380772  
g    lgwatch lgwatch_1                      <NA>                    <NA>  
             lgwatch_2                 50.934812               15.266357  
h    lgw

In [9]:
valid_activities = {
    "bike",
    "sit",
    "stand",
    "walk",
    "stairsup",
    "stairsdown",
}

assert dataset["gt"].notna().all(), "Missing target labels remain."
assert dataset["gt"].isin(valid_activities).all(), (
    "Unexpected target labels remain."
)

print(dataset["gt"].value_counts())

gt
walk          15307
sit           14062
stand         12432
stairsup      11889
bike          11674
stairsdown    10011
Name: count, dtype: Int64


In [10]:
print("Missing activity labels:", dataset["gt"].isna().sum())
print(dataset["gt"].value_counts(dropna=False))

Missing activity labels: 0
gt
walk          15307
sit           14062
stand         12432
stairsup      11889
bike          11674
stairsdown    10011
Name: count, dtype: Int64
